# 02: 多数据集合并 + 跨数据集质量诊断

将各 per-dataset QC 产出合并为统一的 AnnData，并执行跨数据集 QC 诊断。

**合并策略**：
- 基因交集（inner join）：只保留所有数据集共享的基因，确保跨数据集可比性
- cell_id 全局唯一：per-dataset notebook 已确保 ID 不冲突
- 跨数据集诊断：检查合并后各数据集的 QC 分布是否一致，标记异构性

**输出**：合并后的 `02_merged_v1.h5ad`，作为下游 03-15 的统一入口。

In [ ]:
# === PARAMS ===
UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUNS = {  # 键必须是 Stage 01 写入的 canonical source_dataset
    "Nancang_2025": "01-nancang-v1-run001",
    "Kim_2023": "01-kim-v1-run001",
    "Nowicki_2023": "01-nowicki-v1-run001",
    "Yue_2024": "01-yue-v1-run001",
}
RUN_ID = "02-merged-v1-run001"  # 每次调参改用新 ID，禁止覆盖旧 run
RUN_ROOT = "results/runs"
OUTPUT_FILENAME = "02_merged_v1.h5ad"

JOIN_GENES = "inner"
MIN_SHARED_GENES = 15000
BATCH_KEY = "source_dataset"
DOWNSAMPLE_TO_MIN = False
OUTPUT_VERSION = 1
RANDOM_SEED = 42

# 关键 marker 基因——合并后检查这些基因是否在 inner join 交集中
# 如果缺失，说明基因名映射有问题，需要排查
CRITICAL_MARKERS = [
    "EPCAM", "CDH1", "VIM", "PTPRC",  # 大类 compartment
    "MUC5AC", "MUC6", "TFF1", "TFF2",  # 胃黏膜 marker
    "LGR5", "OLFM4",                    # 干细胞
    "MKI67", "TOP2A",                   # 增殖
    "CD3D", "CD4", "CD8A",             # T 细胞
]

In [ ]:
# === Setup ===
import sys, os, json
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

import scanpy as sc
import anndata
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc
from pathlib import Path
from scrna_integration.run_contract import (
    atomic_write_json, collect_runtime_provenance, determine_stage_status, prepare_run,
    promote_run, resume_run, sha256_file, snapshot_effective_parameters, validate_checkpoint,
    validate_expression_contract,
)

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')} | anndata {importlib.metadata.version('anndata')}")

## 加载各 per-dataset 产出

In [ ]:
# 加载每个 per-dataset h5ad：只信 promoted manifest 及其 hash 校验后的 checkpoint
_expected_sources = {"Nancang_2025", "Kim_2023", "Nowicki_2023", "Yue_2024"}
if set(UPSTREAM_RUNS) != _expected_sources:
    raise ValueError(f"UPSTREAM_RUNS 键必须恰为四个 canonical 来源: {_expected_sources}")
if len(set(UPSTREAM_RUNS.values())) != len(UPSTREAM_RUNS):
    raise ValueError("UPSTREAM_RUNS 的 run ID 必须唯一")
PER_DATASET_PATHS = []
upstream_inputs = []
adatas = []
dataset_info = []
for expected_source, upstream_run_id in UPSTREAM_RUNS.items():
    upstream_run = resume_run(UPSTREAM_RUN_ROOT, upstream_run_id, promoted=True)
    manifest_path = upstream_run.promoted_dir / "manifest.json"
    manifest_data = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest_data.get("run_id") != upstream_run_id:
        raise ValueError(f"manifest run_id 与配置不匹配: {expected_source}")
    if manifest_data.get("stage") != "01_qcd":
        raise ValueError(f"manifest stage 必须是 01_qcd: {expected_source}")
    if manifest_data.get("source_dataset") != expected_source:
        raise ValueError(f"manifest source_dataset 与配置不匹配: {expected_source}")
    checkpoint = validate_checkpoint(manifest_path)
    fp = str(checkpoint)
    ad = sc.read_h5ad(fp)
    source_values = sorted(map(str, ad.obs["source_dataset"].dropna().unique())) if "source_dataset" in ad.obs.columns else []
    if source_values != [expected_source] or ad.obs["source_dataset"].isna().any():
        raise ValueError(f"obs source_dataset 与配置不匹配: {expected_source} != {source_values}")
    # 确保基因名唯一——不同数据集的基因名可能因注释版本差异产生重复（如同一个
    # ensembl ID 在两个数据集中映射到了大小写不同的 symbol），anndata.concat
    # 会因 axis 上存在重复标签而报错。var_names_make_unique 是最小防御性修复。
    ad.var_names_make_unique()
    # === 逐来源 assert layers["counts"] 存在 + 非负整数(blockwise) + shape 对齐 ===
    # 决策 2：02 拼接前逐来源检查契约，任一不合格即停止——不靠全局 preprocessing_done 猜状态
    _counts_layer = ad.layers.get("counts")
    if _counts_layer is None:
        raise ValueError(
            f"{expected_source}: layers['counts'] 不存在——上游 01 未建立 counts 契约，"
            f"请重跑对应 01 notebook"
        )
    if _counts_layer.shape != ad.X.shape:
        raise ValueError(
            f"{expected_source}: layers['counts'] shape {_counts_layer.shape} "
            f"与 X shape {ad.X.shape} 不一致"
        )
    # blockwise 非负整数检查：采样 data 数组块，避免全量扫描
    _X_data = _counts_layer.data if sp.issparse(_counts_layer) else _counts_layer.ravel()
    if len(_X_data) == 0:
        raise ValueError(f"{expected_source}: layers['counts'] 为空")
    _rng = np.random.default_rng(42)
    _n_sample = min(len(_X_data), 200000)
    _sample = _rng.choice(_X_data, size=_n_sample, replace=False)
    if (_sample < 0).any():
        raise ValueError(
            f"{expected_source}: layers['counts'] 含负值——不是原始整数 counts"
        )
    _is_int = np.allclose(_sample, np.round(_sample), rtol=0, atol=1e-8)
    if not _is_int:
        _diff = np.abs(_sample - np.round(_sample))
        _bad = _sample[_diff > 1e-8][:5]
        raise ValueError(
            f"{expected_source}: layers['counts'] 含非整数值（blockwise 抽样检查），"
            f"示例: {_bad.tolist()}——可能已被归一化"
        )
    print(f"  {expected_source}: layers['counts'] OK"
          f" (blockwise sampled {_n_sample:,} values,"
          f" non-negative integers, shape={_counts_layer.shape})")
    adatas.append(ad)
    PER_DATASET_PATHS.append(fp)
    upstream_inputs.append({
        "source_dataset": expected_source, "run_id": upstream_run_id, "stage": "01_qcd",
        "manifest_path": str(manifest_path), "manifest_sha256": sha256_file(manifest_path),
        "checkpoint_path": fp, "checkpoint_sha256": sha256_file(checkpoint),
    })
    qc_rpt = ad.uns.get("qc_report_v1", {})
    src = ad.obs["source_dataset"].iloc[0] if "source_dataset" in ad.obs.columns else "?"
    dataset_info.append({"路径": fp, "数据集": src, "细胞数": ad.n_obs, "基因数": ad.n_vars,
                         "QC策略": qc_rpt.get("strategy", "?"), "去除细胞": qc_rpt.get("cells_removed", "?"),
                         "去除比例": f"{qc_rpt.get('pct_removed', '?')}%"})
print("===== 各数据集加载摘要 =====")
display(pd.DataFrame(dataset_info))
print(f"\n共加载 {len(adatas)} 个数据集，总细胞数: {sum(a.n_obs for a in adatas):,}")

## 基因空间交集分析

不同数据集的基因空间通常不完全相同。基因交集过小会导致整合分析丢失大量信息。
若交集 < MIN_SHARED_GENES，可能是某个数据集的基因 ID 体系未正确转换。

In [ ]:
# 基因空间交集分析
gene_sets = {}
for ad in adatas:
    src = ad.obs["source_dataset"].iloc[0]
    gene_sets[src] = set(ad.var_names)
all_union = set.union(*gene_sets.values()) if gene_sets else set()
all_inter = all_union.copy()
for gs in gene_sets.values():
    all_inter &= gs
n_union, n_inter = len(all_union), len(all_inter)
print(f"基因并集: {n_union:,}  交集: {n_inter:,} ({100*n_inter/n_union:.1f}%)")
if n_inter < MIN_SHARED_GENES:
    print(f"WARNING: 共享基因 {n_inter:,} < MIN_SHARED_GENES={MIN_SHARED_GENES:,}")
else:
    print(f"OK 共享基因 {n_inter:,} >= {MIN_SHARED_GENES:,}")

# 关键 marker 交集检查
if CRITICAL_MARKERS:
    missing = [g for g in CRITICAL_MARKERS if g not in all_inter]
    if missing:
        print(f"\nWARNING: {len(missing)} 个关键 marker 不在基因交集中: {missing}")
        # 报告每个缺失 marker 在哪个数据集中缺失
        for gene in missing:
            present_in = [src for src, gs in gene_sets.items() if gene in gs]
            absent_from = [src for src, gs in gene_sets.items() if gene not in gs]
            print(f"  {gene}: 存在于 {present_in}, 缺失于 {absent_from}")
        print("  -> 建议检查缺失数据集的基因名映射（symbol vs ensembl 混淆？）")
    else:
        print(f"\nOK 全部 {len(CRITICAL_MARKERS)} 个关键 marker 均在基因交集中")

# 条形图
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
srcs = list(gene_sets.keys()); counts = [len(gene_sets[s]) for s in srcs]
bars = axes[0].bar(range(len(srcs)), counts, color="steelblue", edgecolor="white")
axes[0].set_xticks(range(len(srcs))); axes[0].set_xticklabels(srcs, rotation=30, ha="right")
axes[0].set_ylabel("基因数"); axes[0].set_title("各数据集基因数")
for b, c in zip(bars, counts):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+200, f"{c:,}", ha="center", fontsize=9)
axes[1].bar(["并集","交集"], [n_union, n_inter], color=["lightgray","steelblue"], edgecolor="white", width=0.4)
axes[1].set_ylabel("基因数"); axes[1].set_title(f"基因交集 ({JOIN_GENES} join)")
for i, v in enumerate([n_union, n_inter]):
    axes[1].text(i, v+200, f"{v:,}", ha="center", fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig("results/figures/02_merged_gene_intersection.png", dpi=150, bbox_inches="tight")
plt.show()

## 基因注释跨数据集一致性诊断

Inner join 丢失的基因可能不是真的缺失，而是同一基因在不同数据集中用了不同名称（大小写不一致、alias 问题）。

In [ ]:
# === 基因注释一致性诊断 ===

# 1. 每个数据集 inner join 丢失了多少基因
print("===== 基因 Inner Join 损失分析 =====")
for src, gs in gene_sets.items():
    lost = gs - all_inter
    print(f"  {src}: {len(lost):,} 基因不在交集中（总 {len(gs):,}，保留率 {100*len(gs & all_inter)/len(gs):.1f}%）")

# 2. 构建基因名 uppercase 索引（供后续 alias 诊断使用）
# 大小写冲突检测已移除——上游 per-dataset notebook 结尾断言基因 ID 大小写不混用（批 1 F3）
all_genes_upper = {}
for src, gs in gene_sets.items():
    for g in gs:
        key = g.upper()
        if key not in all_genes_upper:
            all_genes_upper[key] = set()
        all_genes_upper[key].add((src, g))

# 3. 仅在单个数据集中缺失的基因（alias 候选）
_almost_universal = []
for key, entries in all_genes_upper.items():
    sources_with = set(e[0] for e in entries)
    if len(sources_with) == len(gene_sets) - 1:
        missing_src = (set(gene_sets.keys()) - sources_with).pop()
        _almost_universal.append((key, missing_src))

if _almost_universal:
    print(f"\n注意：{len(_almost_universal):,} 个基因仅在 1 个数据集中缺失（可能是 alias/注释版本问题）:")
    for gene, src in _almost_universal[:10]:
        print(f"  {gene}: 缺失于 {src}")
    if len(_almost_universal) > 10:
        print(f"  ... 共 {len(_almost_universal):,} 个")
else:
    print("\n✓ 无单数据集缺失基因")

## 合并（anndata.concat）

In [ ]:
# 合并所有 per-dataset AnnData
src_keys = [ad.obs["source_dataset"].iloc[0] for ad in adatas]
print(f"合并 {len(adatas)} 个数据集: {src_keys}, join={JOIN_GENES}")
adata = anndata.concat(adatas, join=JOIN_GENES, label="source_dataset", keys=src_keys, index_unique="-")
n_unique = adata.obs_names.nunique()
if n_unique != adata.n_obs:
    print(f"WARNING cell_id 不唯一: {adata.n_obs} obs vs {n_unique} unique")
else:
    print(f"OK cell_id 唯一: {n_unique:,} ids")
print(f"合并后 AnnData: {adata.n_obs:,} cells x {adata.n_vars:,} genes")

# 基因数坍塌预警：inner join 后基因数若远小于输入（<50%），提示可能有基因 ID 未对齐
# 上游 per-dataset 已断言基因 ID 体系一致，此处做兜底预警（不阻断）
_input_gene_counts = [len(gs) for gs in gene_sets.values()]
_min_input = min(_input_gene_counts) if _input_gene_counts else adata.n_vars
_collapse_ratio = adata.n_vars / max(_min_input, 1)
if _collapse_ratio < 0.5:
    print(f"WARNING: concat(join='inner') 后基因数 {adata.n_vars:,} < 最小输入基因数 {_min_input:,} 的 50%"
          f"（保留率 {_collapse_ratio:.1%}），可能存在基因 ID 未对齐问题")
else:
    print(f"基因交集保留率: {_collapse_ratio:.1%}（{adata.n_vars:,} / 最小输入 {_min_input:,}）")

display(adata.obs["source_dataset"].value_counts())
for ad in adatas: del ad
adatas.clear(); gc.collect()


## 数据集平衡度诊断

数据集间细胞数严重不平衡时，下游聚类会被大数据集主导，小数据集的稀有细胞类型几乎不可能形成独立 cluster。


In [ ]:
# === 数据集平衡度诊断 + 下采样决策建议 ===
counts = adata.obs["source_dataset"].value_counts()
imbalance_ratio = counts.max() / max(counts.min(), 1)
print(f"===== 数据集平衡度 =====")
print(counts.to_string())
print(f"\n最大/最小比值: {imbalance_ratio:.1f}x")
print(f"\n决策参考：")
if imbalance_ratio > 10:
    print(f"  ⚠️ 严重不平衡（>10x）：小数据集的信号几乎必然被淹没。")
    print(f"  → 强烈建议启用 DOWNSAMPLE_TO_MIN=True，或在 04 使用 scVI（对不平衡更鲁棒）")
elif imbalance_ratio > 3:
    print(f"  ⚠️ 中度不平衡（>3x）：下游聚类可能偏向大数据集。")
    print(f"  → 建议在 05 聚类时关注小数据集特有的 cluster 是否被分辨出来")
    print(f"  → 可选：DOWNSAMPLE_TO_MIN=True 或 04 使用 scVI + BATCH_KEY='source_dataset'")
else:
    print(f"  ✓ 平衡（<3x）：数据集大小可比，无需特殊处理")

# 饼图
fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90)
ax.set_title("各数据集细胞贡献比例")
plt.tight_layout()
plt.savefig("results/figures/02_merged_dataset_balance.png", dpi=150, bbox_inches="tight")
plt.show()


## 跨数据集 QC 诊断

绘制三个主 QC 指标按 source_dataset 分组的小提琴图，检查跨数据集的一致性。

**看什么**：
- 各数据集的 n_genes / total_counts / pct_mt 分布范围是否大致可比
- 类器官数据集（Kim, Yue）的 MT% 基线是否确实高于组织活检数据集（Nancang, Nowicki）

**关键原则**：跨数据集 QC 的目标不是"让所有分布一样"，而是"理解并记录差异"。
如果某个数据集的 MT% 基线确实偏高（如类器官固有特性），强行对齐反而是科学错误。

In [ ]:
# 跨数据集 QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, metric in enumerate(["n_genes", "total_counts", "pct_counts_mt"]):
    if metric not in adata.obs.columns: continue
    sc.pl.violin(adata, keys=metric, groupby="source_dataset", rotation=30, ax=axes[i], show=False)
    axes[i].set_title(f"{metric}（按 source_dataset）")
plt.tight_layout()
fig.savefig("results/figures/02_merged_qc_by_source.png", dpi=150, bbox_inches="tight")
plt.show()


## 跨数据集摘要表

In [ ]:
# 跨数据集 QC 摘要
print("===== 跨数据集 QC 摘要 =====")
rows = []
for src in sorted(adata.obs["source_dataset"].unique()):
    m = adata.obs["source_dataset"] == src
    sub = adata[m]
    row = {"source_dataset": src, "n_cells": sub.n_obs,
           "median_genes": round(sub.obs["n_genes"].median(), 1),
           "median_umi": round(sub.obs["total_counts"].median(), 1),
           "median_mt_pct": round(sub.obs["pct_counts_mt"].median(), 2)}
    if "predicted_doublet" in sub.obs.columns and sub.obs["predicted_doublet"].notna().any():
        row["doublet_rate_pct"] = round(100 * sub.obs["predicted_doublet"].mean(), 2)
    if "phase" in sub.obs.columns:
        for ph in ["G1","S","G2M"]:
            row[f"phase_{ph}_pct"] = round(100 * (sub.obs["phase"] == ph).mean(), 1)
    rows.append(row)
display(pd.DataFrame(rows))


In [ ]:
# 样本级 PCA 快照——快速检测 outlier 样本
# 如果某个样本在 PCA 空间飞出去 → 可能实验失败/错标/组织类型错误
print("===== 样本级 PCA 快照（outlier 检测）=====")

# 临时做一次快速 PCA（不用 HVG，用 top 2000 变异基因）
_adata_tmp = adata.copy()
# 临时 normalize + log1p（不覆盖原数据）
sc.pp.normalize_total(_adata_tmp, target_sum=1e4)
sc.pp.log1p(_adata_tmp)
sc.pp.highly_variable_genes(_adata_tmp, n_top_genes=min(2000, _adata_tmp.n_vars))
sc.tl.pca(_adata_tmp, n_comps=20, use_highly_variable=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# 守卫：sample_id 列不一定在全部数据集的 obs 中存在
_pca_color_right = []
if "sample_id" in _adata_tmp.obs.columns:
    _pca_color_right.append("sample_id")
if "source_dataset" in _adata_tmp.obs.columns:
    _pca_color_right.append("source_dataset")
# 去重（sample_id 和 source_dataset 可能是同一列）
_pca_color_right = list(dict.fromkeys(_pca_color_right))

sc.pl.pca(_adata_tmp, color="source_dataset", ax=axes[0], show=False, title="PCA by source_dataset")
if _pca_color_right:
    sc.pl.pca(_adata_tmp, color=_pca_color_right[0], ax=axes[1], show=False,
              title=f"PCA by {_pca_color_right[0]}")
else:
    sc.pl.pca(_adata_tmp, ax=axes[1], show=False, title="PCA (no coloring)")

plt.tight_layout()
fig.savefig("results/figures/02_merged_sample_pca_snapshot.png", dpi=150, bbox_inches="tight")
plt.show()

# 检测 outlier：每个样本的 PC1/PC2 中心点，如果离全局中心 > 3 SD → 标记
sample_centers = pd.DataFrame(
    _adata_tmp.obsm["X_pca"][:, :2],
    index=_adata_tmp.obs_names,
    columns=["PC1", "PC2"]
)
sample_centers["sample_id"] = _adata_tmp.obs.get("sample_id", _adata_tmp.obs["source_dataset"]).values
centroids = sample_centers.groupby("sample_id")[["PC1", "PC2"]].mean()
global_center = centroids.mean()
distances = ((centroids - global_center) ** 2).sum(axis=1).apply(np.sqrt)
threshold = distances.mean() + 3 * distances.std()
outliers = distances[distances > threshold]
if len(outliers) > 0:
    print(f"WARNING 疑似 outlier 样本（PC 空间距离 > 3 sigma）: {list(outliers.index)}")
    print(f"  距离: {outliers.to_dict()}")
    print(f"  阈值: {threshold:.2f}")
else:
    print(f"OK 无 outlier 样本（阈值={threshold:.2f}）")

del _adata_tmp; gc.collect()

In [ ]:
# === Per-dataset 细胞类型组成估计 ===
# 14 个 canonical marker 的检出率（表达 > 0 的细胞占比），
# 粗估每个数据集的细胞类型组成概貌
_COMPOSITION_MARKERS = {
    "Epithelial": "EPCAM",
    "Parietal": "ATP4A",
    "Chief": "PGA3",
    "Mucous": "MUC5AC",
    "IM": "CDX2",
    "Immune": "PTPRC",
    "T_cell": "CD3D",
    "B_cell": "CD79A",
    "Myeloid": "CD14",
    "Stromal": "VIM",
    "Endothelial": "PECAM1",
    "Enteroendocrine": "CHGA",
    "Proliferating": "MKI67",
    "Smooth_muscle": "ACTA2",
}

_batch_col = "source_dataset"
if _batch_col in adata.obs.columns:
    print("\n===== Per-dataset 细胞类型组成估计（marker 检出率）=====\n")

    _avail_markers = {name: gene for name, gene in _COMPOSITION_MARKERS.items()
                      if gene in adata.var_names}
    _missing = {name: gene for name, gene in _COMPOSITION_MARKERS.items()
                if gene not in adata.var_names}
    if _missing:
        print(f"  未找到: {_missing}\n")

    # 计算检出率矩阵
    _datasets = adata.obs[_batch_col].unique()
    _detection_matrix = pd.DataFrame(index=_datasets, columns=list(_avail_markers.keys()))

    for ds in _datasets:
        _mask = adata.obs[_batch_col] == ds
        _n_cells = _mask.sum()
        for name, gene in _avail_markers.items():
            _expr = adata[_mask, gene].X
            # 稀疏和 dense 矩阵的 >0 + .sum() 行为一致，无需分支
            _detected = (_expr > 0).sum()
            _detection_matrix.loc[ds, name] = f"{100 * _detected / _n_cells:.0f}%"

    print(_detection_matrix.to_string())
    print(f"\n  解读: 检出率 < 5% 表示该数据集可能缺少该细胞类型")
    print(f"  → 如某数据集壁细胞(ATP4A) < 5%，可能是浅层活检或组织处理问题")

    # 标记异常（某谱系在所有数据集中都 < 5%）
    for name, gene in _avail_markers.items():
        _all_low = all(
            float(_detection_matrix.loc[ds, name].replace("%", "")) < 5
            for ds in _datasets
        )
        if _all_low and name not in ("Enteroendocrine", "Smooth_muscle"):  # 这俩本来就稀有
            print(f"  ⚠️ {name}({gene}) 在所有数据集中检出率 < 5%——可能全部缺失")
else:
    print("source_dataset 列不存在，跳过 per-dataset 组成估计")

## Batch Effect 预判

在 normalization 之前量化 batch effect：如果管家基因（ACTB/GAPDH/B2M）在不同数据集中的表达量差异大，说明 technical batch effect 显著，04 中需要更强的校正。


In [ ]:
# === Batch Effect 预判：管家基因跨数据集表达对比 ===
HOUSEKEEPING_GENES = ["ACTB", "GAPDH", "B2M", "MALAT1", "TMSB4X"]
available_hk = [g for g in HOUSEKEEPING_GENES if g in adata.var_names]

if available_hk:
    print(f"===== Batch Effect 预判（管家基因跨数据集对比）=====")
    import scipy.sparse as sp
    _X_hk = adata[:, available_hk].X
    if sp.issparse(_X_hk):
        _X_hk = _X_hk.toarray()
    hk_expr = pd.DataFrame(_X_hk, columns=available_hk, index=adata.obs_names)
    hk_expr["source_dataset"] = adata.obs["source_dataset"].values
    
    # 每个基因在每个数据集中的中位表达
    medians = hk_expr.groupby("source_dataset")[available_hk].median()
    print(medians.round(1).to_string())
    
    # 计算每个基因的 max/min 比值
    ratios = medians.max() / medians.min().replace(0, 0.1)
    print(f"\n管家基因 max/min 比值:")
    for gene in available_hk:
        ratio = ratios[gene]
        flag = " ⚠️" if ratio > 2 else ""
        print(f"  {gene}: {ratio:.1f}x{flag}")
    
    max_ratio = ratios.max()
    if max_ratio > 3:
        print(f"\n⚠️ 强 batch effect（管家基因差异 >3x）：04 中 Harmony theta ≥ 2 或使用 scVI")
    elif max_ratio > 1.5:
        print(f"\n注意：中度 batch effect。04 中默认 Harmony(theta=2) 应足够")
    else:
        print(f"\n✓ batch effect 轻微：管家基因跨数据集一致。04 可尝试 PCA-only 作为基线")
    
    del hk_expr, _X_hk
else:
    print("跳过 batch effect 预判：管家基因不在数据中")


## Obs 列跨数据集覆盖度 + 值一致性

anndata.concat 对 obs 做 outer join（保留所有列），缺失的用 NaN 填充。下面诊断哪些 metadata 列在部分数据集中缺失，以及同一列在不同数据集中的值是否编码一致。

In [ ]:
# === Obs 列跨数据集覆盖度诊断 ===
coverage = {}
for src in sorted(adata.obs["source_dataset"].unique()):
    mask = adata.obs["source_dataset"] == src
    sub_obs = adata.obs.loc[mask]
    coverage[src] = sub_obs.notna().mean()

coverage_df = pd.DataFrame(coverage).T
_sys_cols = ["source_dataset", "cell_id", "n_genes", "total_counts", "pct_counts_mt",
             "pct_counts_ribo", "doublet_score", "predicted_doublet", "S_score", "G2M_score",
             "phase", "log_complexity", "flag_hb", "pct_counts_hb", "pct_counts_stress",
             "ambient_correction_applied", "sample_id", "disease_system", "project_id"]
_meta_cols = [c for c in coverage_df.columns if c not in _sys_cols and not c.startswith("cell_type_original_")]

# 只显示有差异的列
_diff_cols = [c for c in _meta_cols if coverage_df[c].nunique() > 1 or coverage_df[c].min() < 1.0]
if _diff_cols:
    print("===== Obs 列跨数据集覆盖度（仅显示有差异的 metadata 列）=====")
    display(coverage_df[_diff_cols].round(2))
    missing_combos = []
    for col in _diff_cols:
        for src in coverage_df.index:
            if coverage_df.loc[src, col] == 0:
                missing_combos.append(f"  {col}: {src} 完全缺失")
    if missing_combos:
        print(f"\n⚠️ 以下列在部分数据集中完全缺失（下游按此列分组将排除这些数据集）:")
        for line in missing_combos[:20]:
            print(line)
else:
    print("✓ 所有 metadata 列跨数据集一致（无覆盖度差异）")

# 值一致性检查
print("\n===== 关键 metadata 列值分布（跨数据集）=====")
_key_cols = ["disease", "tissue", "sex", "assay", "batch", "donor_id"]
for col in _key_cols:
    if col not in adata.obs.columns:
        continue
    vals_per_src = {}
    for src in sorted(adata.obs["source_dataset"].unique()):
        mask = adata.obs["source_dataset"] == src
        unique_vals = sorted(adata.obs.loc[mask, col].dropna().unique().tolist())
        if unique_vals:
            vals_per_src[src] = unique_vals[:10]
    if vals_per_src:
        print(f"\n{col}:")
        for src, vals in vals_per_src.items():
            print(f"  {src}: {vals}")
        all_val_sets = [set(v) for v in vals_per_src.values()]
        if len(all_val_sets) >= 2:
            overlap = all_val_sets[0]
            for s in all_val_sets[1:]:
                overlap &= s
            if len(overlap) == 0 and all(len(s) > 0 for s in all_val_sets):
                print(f"  ⚠️ 值完全不重叠！可能是编码不一致（如 'M' vs 'male'），需要 value_mapping 对齐")

## Metadata 对齐（LLM 辅助）

当检测到跨数据集值不一致时，调用 LLM 分析并建议统一的 value_mapping。PI 审核后回填 manifest。

In [ ]:
# === Obs Metadata LLM 辅助对齐 ===
_alignment_issues = []
_key_cols = ["disease", "tissue", "sex", "assay", "batch", "donor_id"]
for col in _key_cols:
    if col not in adata.obs.columns:
        continue
    vals_per_src = {}
    for src in sorted(adata.obs["source_dataset"].unique()):
        mask = adata.obs["source_dataset"] == src
        unique_vals = adata.obs.loc[mask, col].dropna().unique().tolist()
        if unique_vals:
            vals_per_src[src] = unique_vals[:20]
    if len(vals_per_src) >= 2:
        _alignment_issues.append({"column": col, "values_per_dataset": vals_per_src})

if _alignment_issues:
    # 构建 prompt
    prompt = (
        "你是单细胞测序数据整合专家。以下是多个 scRNA-seq 数据集合并后，"
        "obs 列在不同数据集中的值。请分析哪些值表示相同含义但编码不一致，"
        "并为每个列生成统一的 value_mapping。\n\n"
        "要求：\n"
        "1. 将各数据集的非标准值映射到一个统一的标准值\n"
        "2. 标准值用全小写英文（如 male/female, normal/CAG/IM 等）\n"
        "3. 如果某个值无法判断含义，保留原值不做映射\n"
        "4. 输出 JSON 格式：{\"column_name\": {\"原值\": \"标准值\", ...}}\n\n"
        "数据：\n"
    )
    for issue in _alignment_issues:
        prompt += f"\n列名: {issue['column']}\n"
        for src, vals in issue["values_per_dataset"].items():
            prompt += f"  {src}: {vals}\n"
    
    _llm_response = None
    
    # 方式 1：项目 llm_config（.env LLM_GROUP 网关）
    try:
        from scrna_integration.llm_config import load_llm_config, call_llm
        config = load_llm_config()
        if config:
            _llm_response = call_llm(prompt, config=config)
            print("✓ LLM 调用成功（via llm_config 网关）")
    except Exception as e:
        print(f"llm_config 调用失败: {e}")
    
    # 方式 2：fallback 到 claude -p
    if _llm_response is None:
        try:
            import subprocess as _sp
            result = _sp.run(["claude", "-p", prompt], capture_output=True, text=True, timeout=120)
            if result.returncode == 0 and result.stdout.strip():
                _llm_response = result.stdout.strip()
                print("✓ LLM 调用成功（via claude -p fallback）")
            else:
                print(f"claude -p 返回码: {result.returncode}")
        except (FileNotFoundError, _sp.TimeoutExpired) as e:
            print(f"claude -p fallback 失败: {e}")
    
    # 展示结果
    if _llm_response:
        print("\n===== LLM 建议的 Value Mapping =====\n")
        print(_llm_response)
        print("\n" + "=" * 60)
        print("审核上方建议后，将合理的映射回填到各数据集的 manifest.yaml")
        print("的 value_mapping 字段中，重跑 01 stage 即可生效。")
        print("⚠️ 请人工审核！LLM 建议可能有误，特别是领域专有术语。")
    else:
        print("\n⚠️ LLM 调用均失败，请手动将以下 prompt 粘贴给 AI：")
        print("-" * 60)
        print(prompt)
        print("-" * 60)
else:
    print("✓ 关键 metadata 列无明显值不一致问题，无需 LLM 对齐")

## QC 异构性记录

In [ ]:
# 记录跨数据集 QC 异构性
qc_reports = {}
qc_strategies = set()
for src in adata.obs["source_dataset"].unique():
    for fp in PER_DATASET_PATHS:
        if not os.path.exists(fp): continue
        tmp = sc.read_h5ad(fp)
        if tmp.obs["source_dataset"].iloc[0] == src:
            rpt = tmp.uns.get("qc_report_v1", {})
            qc_reports[src] = rpt
            qc_strategies.add(rpt.get("strategy", "unknown"))
            del tmp; break

qc_heterogeneous = len(qc_strategies) > 1
print(f"QC 策略: {qc_strategies}, 异构性: {qc_heterogeneous}")
merge_report = {"n_datasets": len(PER_DATASET_PATHS), "join_genes": JOIN_GENES,
    "n_shared_genes": n_inter, "gene_intersection_pct": round(100*n_inter/n_union, 1),
    "qc_heterogeneous": qc_heterogeneous, "qc_strategies": sorted(qc_strategies), "per_dataset_qc": qc_reports}
adata.uns["merge_report_v1"] = merge_report
for k, v in merge_report.items():
    if k != "per_dataset_qc": print(f"  {k}: {v}")
gc.collect()

## 可选：按最小细胞数下采样

In [ ]:
# 可选下采样
if DOWNSAMPLE_TO_MIN:
    min_n = min(adata.obs["source_dataset"].value_counts())
    print(f"下采样到 {min_n} per dataset...")
    np.random.seed(RANDOM_SEED)
    keep_idx = []
    for src in adata.obs["source_dataset"].unique():
        src_idx = adata.obs_names[adata.obs["source_dataset"] == src]
        keep_idx.extend(np.random.choice(src_idx, size=min_n, replace=False))
    adata = adata[keep_idx].copy()
    print(f"下采样后: {adata.n_obs:,} cells")
else:
    print("DOWNSAMPLE_TO_MIN=False")


### Stage 02 Verdict

本 stage 完成后应确认：
- [ ] 各数据集基因交集覆盖率 > 80%
- [ ] QC 分布无极端异常值数据集
- [ ] Housekeeping 批次效应已识别（stage 04 将处理）
- [ ] 元数据列对齐完成（LLM 辅助或手动）


In [ ]:
# Checkpoint：验证后写入不可覆盖 run，并原子提升
_expected_sources = {"Nancang_2025", "Kim_2023", "Nowicki_2023", "Yue_2024"}
_upstream_run_ids = [item.get("run_id") for item in upstream_inputs]
_required_input_fields = {"source_dataset", "run_id", "stage", "manifest_path", "manifest_sha256", "checkpoint_path", "checkpoint_sha256"}
_upstream_provenance_complete = all(
    _required_input_fields <= item.keys() and item["stage"] == "01_qcd" and all(isinstance(item[key], str) and item[key] for key in _required_input_fields)
    and all(Path(item[path_key]).is_file() and sha256_file(item[path_key]) == item[hash_key]
            for path_key, hash_key in (("manifest_path", "manifest_sha256"), ("checkpoint_path", "checkpoint_sha256"))) for item in upstream_inputs
)
hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "x_sparse_float32": sp.issparse(adata.X) and adata.X.dtype == np.float32,
    "four_upstream_inputs_complete": len(upstream_inputs) == 4 and {item.get("source_dataset") for item in upstream_inputs} == _expected_sources and _upstream_provenance_complete,
    "upstream_run_ids_unique": len(set(_upstream_run_ids)) == len(_upstream_run_ids) == 4,
}
stage_status = determine_stage_status({}, hard_postconditions, allow_no_required_methods=True)
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("PER_DATASET_PATHS",), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, ("anndata", "scanpy", "numpy", "pandas", "scipy")
)
manifest_payload = {
    "run_id": RUN_ID, "stage": "02_merged", "stage_status": stage_status.value,
    "inputs": upstream_inputs, "effective_parameters": effective_parameters,
    "runtime_provenance": runtime_provenance, "hard_postconditions": hard_postconditions,
}
run_paths = prepare_run(RUN_ROOT, RUN_ID)
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 02 FAILED: {hard_postconditions}")
adata.uns["stage"] = "02_merged"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = PER_DATASET_PATHS
adata.uns["upstream_inputs"] = upstream_inputs
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
# === expression_contract（决策 1/2）===
# 02：合并后 counts 来自上游 01 建立的 layers["counts"]，concat 拼接后保持不变
# processing_history 从 upstream_inputs 逐来源构建
_processing_history = [
    {
        "source_dataset": item["source_dataset"],
        "upstream_run_id": item["run_id"],
        "checked_at": "02_merged",
    }
    for item in upstream_inputs
]
adata.uns["expression_contract"] = {
    "x_scale": "raw_counts",
    "counts_layer": "counts",
    "counts_source": "layers[counts]",
    "counts_validated": True,
    "counts_integer_check": "blockwise",
    "soupx_layer": None,
    "processing_history": _processing_history,
    "stage": "02",
}
validate_expression_contract(adata, expected_scale="raw_counts", stage="02")
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
OUTPUT_PATH = str(promote_run(run_paths))
print(f"OK 提升 {OUTPUT_PATH}  ({adata.n_obs} cells x {adata.n_vars} genes)")
del adata
gc.collect()